In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import yfinance as yf
from arch import arch_model
import os

In [ ]:
TRADING_DAYS = 245

# All trading days from 2025-12-01 through today
#CALC_DATES = pd.bdate_range(start='2025-12-01', end=pd.Timestamp.today().normalize())
#CALC_DATES = pd.bdate_range(start='2025-12-01', end=pd.Timestamp.today().normalize())
CALC_DATES = pd.bdate_range(start=pd.Timestamp.today().normalize(), end=pd.Timestamp.today().normalize())

# Keep scalar CALC_DATE for existing downstream code
CALC_DATE = CALC_DATES.max()

#PRICE_START_DATE = CALC_DATE - pd.tseries.offsets.BDay(TRADING_DAYS)
#PRICE_END_DATE = CALC_DATE
STRESS_DATE_START = pd.to_datetime('2008-07-01')
STRESS_DATE_END = pd.to_datetime('2009-03-31')

# Accepts int or list, e.g. 10 or [1, 10]
SIM_DAYS = [1, 10]
SIM_DAYS_LIST = [SIM_DAYS] if isinstance(SIM_DAYS, int) else list(SIM_DAYS)
N_SIMS = 1_000

In [3]:
for date in CALC_DATES:
    PRICE_START_DATE = date - pd.tseries.offsets.BDay(TRADING_DAYS)
    PRICE_END_DATE = date + pd.Timedelta(days=1)
    #setup output directory
    date_str = date.strftime('%Y%m%d')
    output_dir = f'outputs/{date_str}'

    os.makedirs(output_dir, exist_ok=True)



    #load portfolio and get positions
    portfolio_df = pd.read_csv("portfolio.csv")
    portfolio_df['ticker'] = portfolio_df['ticker'].astype(str)
    tickers = portfolio_df['ticker'].unique().tolist()

    positions_df = portfolio_df.copy()

    positions_df['purchase_date'] = pd.to_datetime(positions_df['purchase_date'])
    positions_df = positions_df[positions_df["purchase_date"] <= date]

    positions_df['signed_qty'] = positions_df['quantity']
    positions_df.loc[positions_df['buy_sell'] == 'sell', 'signed_qty'] *= -1

    positions = positions_df.groupby("ticker")["signed_qty"].sum()
    positions = positions[positions != 0]



    # get price data
    price_data = yf.download(
        tickers=positions.index.tolist(),
        start=PRICE_START_DATE,
        end=PRICE_END_DATE,
        group_by='ticker',
        auto_adjust=True,
        progress=False
    )

    price_data = price_data.loc[:, (slice(None), ['Close', 'Volume'])]
    close_prices = price_data.xs('Close', level=1, axis=1)
    log_returns = np.log(close_prices / close_prices.shift(1)).dropna()

    stress_data = yf.download(
        tickers=positions.index.tolist(),
        start=STRESS_DATE_START,
        end=STRESS_DATE_END,
        group_by='ticker',
        auto_adjust=True,
        progress=False
    )

    stress_data = stress_data.loc[:, (slice(None), ['Close', 'Volume'])]
    stress_close_prices = stress_data.xs('Close', level=1, axis=1).reindex(columns=close_prices.columns)
    stress_log_returns = np.log(stress_close_prices / stress_close_prices.shift(1)).dropna()



    def simulate_returns(log_returns, sim_days):
        garch_params = {}
        tickers = log_returns.columns

        for ticker in tickers:
            r_pct = log_returns[ticker] * 100

            model = arch_model(
                r_pct,
                vol='Garch',
                p=1,
                q=1,
                mean='Constant',
                dist='normal'
            )

            res = model.fit(disp='off')
            params = res.params
            garch_params[ticker] = {
                'mu': params['mu'] / 100,
                'omega': params['omega'],
                'alpha': params['alpha[1]'],
                'beta': params['beta[1]'],
                'sigma_pct': res.conditional_volatility.iloc[-1]
            }

        n_assets = len(tickers)
        corr = log_returns.corr().values
        L = np.linalg.cholesky(corr)
        sim_returns = np.zeros((N_SIMS, sim_days, n_assets))
        mu_vec = np.array([garch_params[t]['mu'] for t in tickers])

        for sim in range(N_SIMS):
            sigma2_pct = np.array([garch_params[t]['sigma_pct'] ** 2 for t in tickers])

            for step in range(sim_days):
                z = np.random.normal(size=n_assets)
                z_corr = L @ z
                sigma_pct = np.sqrt(sigma2_pct)
                epsilon_pct = sigma_pct * z_corr
                sim_returns[sim, step, :] = mu_vec + (epsilon_pct / 100)

                for i, ticker in enumerate(tickers):
                    p = garch_params[ticker]
                    sigma2_pct[i] = (
                        p['omega']
                        + p['alpha'] * (epsilon_pct[i] ** 2)
                        + p['beta'] * sigma2_pct[i]
                    )

        return sim_returns

    def get_pnls(positions, close_prices, sim_returns):
        n_assets = sim_returns.shape[2]

        # get final pnls
        S0 = close_prices.iloc[-1].values.reshape(1, -1)
        cum_log_returns = sim_returns.sum(axis=1)  # shape: (n_sims, n_assets)
        final_prices = S0 * np.exp(cum_log_returns)
        final_prices.shape = (N_SIMS, n_assets)
        positions_vec = positions.reindex(close_prices.columns).fillna(0).values
        asset_pnl = (final_prices - S0) * positions_vec
        pnl = asset_pnl.sum(axis=1)

        # get risk metrics
        VaR_95 = np.percentile(pnl, 5)
        VaR_99 = np.percentile(pnl, 1)
        ES_95 = pnl[pnl <= VaR_95].mean()
        ES_99 = pnl[pnl <= VaR_99].mean()

        # return
        #return [pnl, VaR_95, VaR_99, ES_95]
        return {
            "pnls": pnl,
            "VaR_95": VaR_95,
            "VaR_99": VaR_99,
            "ES_95": ES_95,
            "ES_99": ES_99
        }




    # export current portfolio breakdown for the valuation date
    latest_prices = close_prices.iloc[-1].reindex(positions.index)
    holdings_df = positions.rename('quantity').reset_index()
    holdings_df = holdings_df.merge(
        company_lookup,
        on='ticker',
        how='left'
    )
    holdings_df['price'] = holdings_df['ticker'].map(latest_prices)
    holdings_df['market_value'] = holdings_df['quantity'] * holdings_df['price']
    holdings_df['weight'] = holdings_df['market_value'] / holdings_df['market_value'].sum()
    holdings_df = holdings_df[['ticker', 'company_name', 'quantity', 'price', 'market_value', 'weight']]
    holdings_df.to_csv(f'{output_dir}/portfolio_breakdown.csv', index=False)

    stock_prices_df = holdings_df[['ticker', 'company_name', 'price']].copy()
    stock_prices_df.to_csv(f'{output_dir}/stock_prices.csv', index=False)

    # get base risk and pnls for each simulation horizon
    date_str = date.strftime('%Y%m%d')
    portfolio_value = holdings_df['market_value'].sum()
    pnl_rows = []
    risk_rows = []
    marginal_risk_rows = []

    for sim_days in SIM_DAYS_LIST:
        sim_returns = simulate_returns(log_returns, sim_days)
        all_pnls = get_pnls(positions, close_prices, sim_returns)
        pnl_rows.extend({
            'sim_days': sim_days,
            'sim_index': sim_index,
            'pnl': pnl
        } for sim_index, pnl in enumerate(all_pnls['pnls'], start=1))

        risk_rows.append({
            'scenario': 'normal',
            'sim_days': sim_days,
            'Portfolio Value': portfolio_value,
            'VaR_95': all_pnls['VaR_95'],
            'VaR_99': all_pnls['VaR_99'],
            'ES_95': all_pnls['ES_95'],
            'ES_99': all_pnls['ES_99']
        })

        for ticker in positions.index:
            positions_excluded = positions.copy()
            positions_excluded.loc[ticker] = 0
            excluded_results = get_pnls(positions_excluded, close_prices, sim_returns)

            marginal_risk_rows.append({
                'sim_days': sim_days,
                'excluded_ticker': ticker,
                'VaR_95': excluded_results['VaR_95'],
                'VaR_99': excluded_results['VaR_99'],
                'ES_95': excluded_results['ES_95'],
                'ES_99': excluded_results['ES_99']
            })

        stress_sim_returns = simulate_returns(stress_log_returns, sim_days)
        stress_pnls = get_pnls(positions, close_prices, stress_sim_returns)
        risk_rows.append({
            'scenario': 'stress',
            'sim_days': sim_days,
            'Portfolio Value': portfolio_value,
            'VaR_95': stress_pnls['VaR_95'],
            'VaR_99': stress_pnls['VaR_99'],
            'ES_95': stress_pnls['ES_95'],
            'ES_99': stress_pnls['ES_99']
        })

    pnls_df = pd.DataFrame(pnl_rows)
    risk_df = pd.DataFrame(risk_rows)
    marginal_risk_df = pd.DataFrame(marginal_risk_rows)

    pnls_df.to_csv(f'{output_dir}/pnls.csv', index=False)
    risk_df.to_csv(f'{output_dir}/risk.csv', index=False)
    marginal_risk_df.to_csv(f'{output_dir}/marginal_risk.csv', index=False)
    print("Completed calculations for date:", date_str)

NameError: name 'company_lookup' is not defined